In [2]:
# Embeddings model and Chat
from langchain_openai import OpenAIEmbeddings
# Prompt template
from langchain_core.documents import Document
# Vector database
from langchain_chroma import Chroma

import os
import psycopg2
import datetime, uuid
import numpy as np
from decimal import Decimal
from datetime import datetime

/Users/gblasd/Documents/SmartBnB/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths

In [3]:
CHROMA_PATH = os.path.abspath(os.path.join("..", "db", "chroma_db"))

In [4]:
CHROMA_PATH

'/Users/gblasd/Documents/SmartBnB/db/chroma_db'

## Utils

In [5]:
# Create connection to the database and initialize it
def create_db_connection() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5433"),
        dbname=os.getenv("DB_NAME", "smartbnb"),
        user=os.getenv("DB_USER", "admin"),
        password=os.getenv("DB_PASSWORD", "admin")
    )
    return conn

def _sanitize_metadata_value(v):
    # to contain only str, int, float, or bool
    if isinstance(v, (str, int, float, bool)):
        return v
    elif v is None:
        return None
    elif isinstance(v, (Decimal,)):
        return float(v)
    elif isinstance(v, (list, tuple)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, dict):
        return {k: _sanitize_metadata_value(v) for k, v in v.items()}
    elif isinstance(v, (set, frozenset)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, (np.ndarray,)):
        return v.tolist()
    elif isinstance(v, (datetime.datetime, datetime.date)):
        return v.isoformat()
    elif isinstance(v, uuid.UUID):
        return str(v)
    return v


def drop_connection(conn):
    conn.close()


def _sanitize_metadata_value(v):
    # to contain only str, int, float, or bool
    if isinstance(v, (str, int, float, bool)):
        return v
    elif v is None:
        return None
    elif isinstance(v, (Decimal,)):
        return float(v)
    elif isinstance(v, (list, tuple)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, dict):
        return {k: _sanitize_metadata_value(v) for k, v in v.items()}
    elif isinstance(v, (set, frozenset)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, (np.ndarray,)):
        return v.tolist()
    elif isinstance(v, (datetime.datetime, datetime.date)):
        return v.isoformat()
    elif isinstance(v, uuid.UUID):
        return str(v)
    return v



## Get Data 

In [6]:
def get_documents_from_pg() -> list:

    # return list of Documents
    documents = []

    # Create DB connection
    conn = create_db_connection()
    
    # Define query
    query = """select l.id, l.listing_url, l.name, l.description, l.neighborhood_overview, l.neighbourhood_cleansed,
        l.property_type, l.room_type, l.accommodates, l.bathrooms, l.bathrooms_text, l.bedrooms, 
        l.beds, l.amenities, l.price, l.latitude, l.longitude, l.minimum_nights, l.maximum_nights, 
        l.has_availability, l.review_scores_accuracy, l.review_scores_communication,
        l.review_scores_cleanliness, l.review_scores_location, l.review_scores_value, 
        l.review_scores_rating, l.reviews_per_month, l.instant_bookable,
        l.calculated_host_listings_count, l.calculated_host_listings_count_entire_homes,
        l.calculated_host_listings_count_private_rooms, l.calculated_host_listings_count_shared_rooms
    from public.listings l
    where l.has_availability is true limit 100"""

    metadata_keys = ["neighbourhood_cleansed","property_type",
                        "room_type", "bathrooms", "bathrooms_text", "bedrooms", "beds",
                        "price", "latitude", "longitude", "minimum_nights", 
                        "maximum_nights", "has_availability", 
                        "review_scores_accuracy", "amenities"]
    
    # query data
    with conn.cursor() as cur:
        cur.execute(query)
        records = cur.fetchall()

        if not records:
            print("No documents!!!")
            return

        for record in records:
            # Add the record to the vector database collection
            row = {}
            row["metadata"] = [
                {
                    col.name: _sanitize_metadata_value(record[i]) 
                    for i, col in enumerate(cur.description) 
                        if col.name  in metadata_keys
                        
                }
            ]

            # Convert amenities from string to list
            if "amenities" in row["metadata"][0]:
                amenities_str = row["metadata"][0]["amenities"]
                amenities_list = [amenity.strip() for amenity in amenities_str.split(",")]
                row["metadata"][0]["amenities"] = amenities_list

            # create Document object            
            doc = Document(
                page_content=str(record[3]),
                metadata=row["metadata"][0],
                id=record[0]
            )

            documents.append(doc)

    # Drop connection
    drop_connection(conn)

    return documents

In [7]:
documents = get_documents_from_pg()

In [8]:
documents

[Document(id='728372', metadata={'neighbourhood_cleansed': 'Cuauhtémoc', 'property_type': 'Entire home', 'room_type': 'Entire home/apt', 'bathrooms': 5.5, 'bathrooms_text': '5.5 baths', 'bedrooms': 10.0, 'beds': 17.0, 'amenities': ['Dedicated workspace', 'Smoking allowed', 'Oven', 'Pets allowed', 'Hair dryer', 'Iron', 'Essentials', 'Dishes and silverware', 'Hot water', 'TV with standard cable', 'Fire extinguisher', 'Host greets you', 'Private patio or balcony', 'Cooking basics', 'Kitchen', 'Bathtub', 'Private backyard \\u2013 Fully fenced', 'Paid parking off premises', 'Private entrance', 'Washer', 'Wifi', 'Hangers', 'Stove', 'Refrigerator', 'Coffee maker', 'City skyline view', 'Children\\u2019s dinnerware', 'Body soap', 'Microwave', 'Exterior security cameras on property', 'Free street parking'], 'price': 3507.0, 'latitude': 19.44, 'longitude': -99.14, 'minimum_nights': 1.0, 'maximum_nights': 90.0, 'has_availability': True, 'review_scores_accuracy': 4.79}, page_content='IF YOU HAVE A 

## Embeddings

In [9]:
# Initialize the OpenAI embedding model
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=256 
)

## Store Documents in Chroma

In [10]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings_model,
    persist_directory=CHROMA_PATH,
    collection_name='smartbnb_vector_store'
)

print(f"Successfully embedded and stored {len(documents)} documents in ChromaDB at {CHROMA_PATH}.")

Successfully embedded and stored 100 documents in ChromaDB at /Users/gblasd/Documents/SmartBnB/db/chroma_db.


## Query vector store

In [11]:
vector_store.similarity_search('coyoacan', k=3)

[Document(id='18544530', metadata={'minimum_nights': 1.0, 'bathrooms_text': '1 private bath', 'review_scores_accuracy': 4.5, 'has_availability': True, 'property_type': 'Room in boutique hotel', 'latitude': 19.36, 'maximum_nights': 1125.0, 'beds': 0.0, 'room_type': 'Private room', 'bedrooms': 1.0, 'longitude': -99.16, 'bathrooms': 1.0, 'amenities': ['Carbon monoxide alarm', 'Breakfast', 'First aid kit', 'HDTV with Fire TV', 'Hangers', 'Kitchen', 'Keypad', 'Wifi', 'Smoke alarm', 'Self check-in', 'Shampoo'], 'price': 1606.0, 'neighbourhood_cleansed': 'Coyoacán'}, page_content="It's a very adoco colonial type in the coyoacan area, new and comfortable"),
 Document(id='70644', metadata={'property_type': 'Entire rental unit', 'price': 2318.0, 'amenities': ['Shampoo', 'Carbon monoxide alarm', 'Dedicated workspace', 'Mini fridge', 'Hair dryer', 'Drying rack for clothing', 'Shower gel', 'Varies conditioner', 'First aid kit', 'Iron', 'Cleaning available during stay', 'Essentials', 'Dishes and sil

In [12]:
[
    document.to_json()['kwargs'] 
    for document
    in vector_store.similarity_search(
        query='coyoacan', k=3,
        )
]

[{'id': '18544530',
  'metadata': {'bathrooms_text': '1 private bath',
   'review_scores_accuracy': 4.5,
   'minimum_nights': 1.0,
   'room_type': 'Private room',
   'bathrooms': 1.0,
   'bedrooms': 1.0,
   'maximum_nights': 1125.0,
   'property_type': 'Room in boutique hotel',
   'beds': 0.0,
   'has_availability': True,
   'neighbourhood_cleansed': 'Coyoacán',
   'amenities': ['Carbon monoxide alarm',
    'Breakfast',
    'First aid kit',
    'HDTV with Fire TV',
    'Hangers',
    'Kitchen',
    'Keypad',
    'Wifi',
    'Smoke alarm',
    'Self check-in',
    'Shampoo'],
   'longitude': -99.16,
   'latitude': 19.36,
   'price': 1606.0},
  'page_content': "It's a very adoco colonial type in the coyoacan area, new and comfortable",
  'type': 'Document'},
 {'id': '70644',
  'metadata': {'bathrooms': 1.0,
   'has_availability': True,
   'longitude': -99.16,
   'latitude': 19.35,
   'property_type': 'Entire rental unit',
   'beds': 1.0,
   'room_type': 'Entire home/apt',
   'bedrooms': 

In [13]:
vector_store.similarity_search_with_score('coyoacan', k=3)

[(Document(id='18544530', metadata={'maximum_nights': 1125.0, 'neighbourhood_cleansed': 'Coyoacán', 'review_scores_accuracy': 4.5, 'has_availability': True, 'longitude': -99.16, 'property_type': 'Room in boutique hotel', 'latitude': 19.36, 'bathrooms': 1.0, 'amenities': ['Carbon monoxide alarm', 'Breakfast', 'First aid kit', 'HDTV with Fire TV', 'Hangers', 'Kitchen', 'Keypad', 'Wifi', 'Smoke alarm', 'Self check-in', 'Shampoo'], 'room_type': 'Private room', 'bedrooms': 1.0, 'price': 1606.0, 'bathrooms_text': '1 private bath', 'beds': 0.0, 'minimum_nights': 1.0}, page_content="It's a very adoco colonial type in the coyoacan area, new and comfortable"),
  0.7956737279891968),
 (Document(id='70644', metadata={'bathrooms': 1.0, 'maximum_nights': 180.0, 'property_type': 'Entire rental unit', 'bathrooms_text': '1 bath', 'bedrooms': 1.0, 'beds': 1.0, 'has_availability': True, 'neighbourhood_cleansed': 'Coyoacán', 'room_type': 'Entire home/apt', 'longitude': -99.16, 'price': 2318.0, 'review_sco

In [14]:
query = vector_store.similarity_search_with_score('coyoacan', k=3)

In [15]:

query[0][0].to_json()

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'document', 'Document'],
 'kwargs': {'id': '18544530',
  'metadata': {'neighbourhood_cleansed': 'Coyoacán',
   'has_availability': True,
   'price': 1606.0,
   'beds': 0.0,
   'latitude': 19.36,
   'bedrooms': 1.0,
   'minimum_nights': 1.0,
   'bathrooms': 1.0,
   'room_type': 'Private room',
   'maximum_nights': 1125.0,
   'longitude': -99.16,
   'review_scores_accuracy': 4.5,
   'bathrooms_text': '1 private bath',
   'amenities': ['Carbon monoxide alarm',
    'Breakfast',
    'First aid kit',
    'HDTV with Fire TV',
    'Hangers',
    'Kitchen',
    'Keypad',
    'Wifi',
    'Smoke alarm',
    'Self check-in',
    'Shampoo'],
   'property_type': 'Room in boutique hotel'},
  'page_content': "It's a very adoco colonial type in the coyoacan area, new and comfortable",
  'type': 'Document'}}

In [26]:
len(vector_store.get(include=["documents"])["documents"])

200